<h1 style="text-align: center;">Data Preparation</h1>
<h3 style="text-align: center;">Bank Marketing Campaign</h3>

---

<h5 style="text-align: right;">By Elmar leonard & Nadya Divia Go</h5>

# **Section 0: Setup**

## **0.1. Import Library**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.feature_selection import SelectKBest

from sklearn.preprocessing import OneHotEncoder, RobustScaler
from feature_engine.outliers import Winsorizer

## **0.2. Global Configuration**

In [ ]:
RANDOM_STATE = 42
pd.set_option('display.max_columns', None)

df =  pd.read_csv(r"..\\data\\interim\\cleaned_data_bank_marketing_campaign.csv")

# **Section 1: Machine Learning Data Preparation**

## **1.1 Initialization**

In [ ]:
feature = df.drop(columns="deposit")
target = df["deposit"].map({"yes": 1, "no": 0})

This step separates the dataset into independent variables (**features**) and the dependent variable (**target**), while preparing the data for model training:

* **What it does:** Splits the DataFrame into `feature` (all columns except `deposit`) and `target` (the `deposit` column mapped from `"yes"/"no"` strings to numeric `1`/`0` values).
* **Why it matters:** 
  * **Model Compatibility:** Scikit-learn estimators require numeric targets to compute losses and train effectively.
  * **Data Leakage Prevention:** Isolating the target early guarantees it won't be accidentally processed during downstream pipeline stages like feature scaling, imputation, or categorical encoding.


## **1.2 Constructing `Training` and `Testing` Data**

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(feature, target, test_size=0.2, random_state=RANDOM_STATE)

x_train.reset_index(inplace=True, drop=True)
y_train.reset_index(inplace=True, drop=True)
x_test.reset_index(inplace=True, drop=True)
y_test.reset_index(inplace=True, drop=True)

This step partitions the dataset into independent training and validation sets, ensuring data integrity for downstream preprocessing:

* **What it does:** Performs an 80/20 train-test split and applies `.reset_index(drop=True)` to all four resulting DataFrames (`X_train`, `X_test`, `y_train`, `y_test`).
* **Why the split matters:** 
  * **Data Allocation:** Splitting the ~7,805 rows yields 6,244 training rows and 1,561 test rows, balancing model optimization with stable generalization tracking.
  * **Reproducibility:** A fixed `random_state=42` guarantees identical data splits across every run.
* **Why resetting the index matters:** 
  * **Preventing Misalignment:** Scikit-learn shuffles row indices during splitting. 
  * **Bug Prevention:** Resetting to a clean `0..N-1` range ensures subsequent operations (like combining one-hot encoded columns via `pd.concat`) align perfectly without introducing silent data shifts or `NaN` errors.



## **1.3 Handling Imbalanced Data**

In [ ]:
print("--- Absolute Counts ---")
print(target.value_counts())

print("\n--- Relative Percentages ---")
print(target.value_counts(normalize=True) * 100)

--- Absolute Counts ---
deposit
0    4075
1    3730
Name: count, dtype: int64

--- Relative Percentages ---
deposit
0    52.210122
1    47.789878
Name: proportion, dtype: float64


This diagnostic step checks the distribution of the target variable to determine if any resampling interventions are required:

* **What it does:** Analyzes the class proportions of the `deposit` target variable before modeling.
* **The Result:** The target is nearly evenly split, with 4,075 "no" values (52.2%) and 3,730 "yes" values (47.8%).
* **Why it matters:** 
  * **Resampling Avoided:** Severe imbalances (e.g., 95/5) require techniques like SMOTE or undersampling. Applying them to a clean 52/48 split would introduce synthetic noise or discard valid data unnecessarily.
  * **Safer Alternatives:** Because the dataset is inherently balanced, heavy-handed data manipulation is bypassed. If minor adjustments are needed during hyperparameter tuning, using `class_weight='balanced'` inside the model grid provides a much safer, less intrusive option.


## **1.4 Data Transformation**

### **1.4.1 Feature Engineering**

In [ ]:
x_train['pdays_contacted'] = (x_train['pdays'] != -1).astype(int)

x_train['age_group'] = pd.cut(
    x_train['age'],
    bins=[17, 30, 40, 50, 60, 96],
    labels=['18-30', '31-40', '41-50', '51-60', '60+']
)

rare_jobs = ['entrepreneur', 'housemaid', 'unknown']
x_train['job_grouped'] = x_train['job'].where(~x_train['job'].isin(rare_jobs), 'other')

x_train.drop(columns="job", inplace=True)

x_train['balance_negative'] = (x_train['balance'] < 0).astype(int)
x_train['balance_log'] = np.log1p(x_train['balance'].clip(lower=0))

display(x_train.head())

,age,balance,housing,loan,contact,month,campaign,pdays,poutcome,pdays_contacted,age_group,job_grouped,balance_negative,balance_log
0,21,216,no,no,cellular,aug,1,-1,unknown,0,18-30,student,0,5.379897
1,30,1599,no,no,cellular,feb,2,-1,unknown,0,18-30,services,0,7.377759
2,61,967,no,no,cellular,aug,1,-1,unknown,0,60+,management,0,6.875232
3,30,3137,yes,no,cellular,jul,7,-1,unknown,0,18-30,self-employed,0,8.051341
4,35,3160,yes,no,cellular,nov,2,95,failure,1,31-40,technician,0,8.058644


This step builds five specialized features on the training set (`x_train`) to capture complex, non-linear relationships and reduce data sparsity:

* **What it does:** Generates five engineered columns:
  * `pdays_contacted`: A binary indicator showing whether a customer was reached in a previous campaign (`pdays != -1`).
  * `age_group`: Bins the continuous `age` column into 5 distinct groups to handle non-linear correlations with the target.
  * `job_grouped`: Collapses the sparsest categories (`entrepreneur`, `housemaid`, `unknown`) into a single `"other"` designation.
  * `balance_negative` / `balance_log`: Separates negative balances into a dedicated flag, then applies a log transformation to the remaining non-negative values to mitigate severe statistical skew.
* **Column Retention Logic:**
  * **Dropped Columns:** The original `job` column is removed because `job_grouped` acts as a direct, cleaner replacement. Keeping both would introduce perfect redundancy.
  * **Retained Columns:** The raw `age`, `pdays`, and `balance` columns are deliberately kept alongside the new variables. While the engineered features simplify the data for the model, the raw, continuous variables provide granular information (like exact financial amounts or exact elapsed days) that might still contain valuable predictive power.



### **1.4.2 Encoding**

In [ ]:
# One Hot Encoder
cat_col = ["contact", "month", "poutcome", "age_group", "job_grouped", "housing", "loan"]

encoder = OneHotEncoder(drop="first",sparse_output=False, handle_unknown='ignore')

encoded_array = encoder.fit_transform(x_train[cat_col])

encoded_col_names = encoder.get_feature_names_out(cat_col)

encoded_df = pd.DataFrame(encoded_array, columns=encoded_col_names, index=x_train.index)

x_train_numeric = x_train.drop(columns=cat_col)
x_train_encoded = pd.concat([x_train_numeric, encoded_df], axis=1)

display(x_train_encoded.head())

,age,balance,campaign,pdays,pdays_contacted,balance_negative,balance_log,contact_telephone,contact_unknown,month_aug,month_dec,month_feb,month_jan,month_jul,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,poutcome_other,poutcome_success,poutcome_unknown,age_group_31-40,age_group_41-50,age_group_51-60,age_group_60+,job_grouped_blue-collar,job_grouped_management,job_grouped_other,job_grouped_retired,job_grouped_self-employed,job_grouped_services,job_grouped_student,job_grouped_technician,job_grouped_unemployed,housing_yes,loan_yes
0,21,216,1,-1,0,0,5.379897,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,30,1599,2,-1,0,0,7.377759,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,61,967,1,-1,0,0,6.875232,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,30,3137,7,-1,0,0,8.051341,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4,35,3160,2,95,1,0,8.058644,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0


This step transforms the categorical attributes into numeric dummy variables and integrates them back into the feature matrix:

* **What it does:** One-hot encodes 7 categorical features (`contact`, `month`, `poutcome`, `age_group`, `job_grouped`, `housing`, `loan`) and concatenates the resulting binary columns with the numeric dataset.
* **Why `drop="first"` is used:** 
  * **Multicollinearity Prevention:** Dropping the first dummy column establishes a reference baseline level and avoids the "dummy variable trap" (perfect correlation). 
  * **Model Safety:** While mandatory for linear and logistic regression models, this structure remains fully compatible with tree-based models like Gradient Boosting.
* **Why `handle_unknown="ignore"` is used:** 
  * **Production Stability:** Because the encoder is fit strictly on `x_train`, any novel category appearing in `x_test` or production data will simply have all its dummy columns set to `0` instead of triggering a fatal runtime error.
* **Pipeline Verification Note:**
  * **Granularity Check:** The `month` feature is encoded here at full 12-month granularity. The 3-tier `month_tier` (low/medium/high) feature explored during EDA is not applied. If the intention is to give tree models the freedom to find their own splits, this is correct; otherwise, verify if `month_tier` was supposed to replace `month`.



### **1.4.3 Winsorization**

In [ ]:
heavily_skewed = ["balance", "campaign", "pdays"]

winsorizer = Winsorizer(tail="both", capping_method="iqr", fold=3, variables=heavily_skewed)

x_train_capped = winsorizer.fit_transform(x_train_encoded)

display(x_train_capped.head())

,age,balance,campaign,pdays,pdays_contacted,balance_negative,balance_log,contact_telephone,contact_unknown,month_aug,month_dec,month_feb,month_jan,month_jul,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,poutcome_other,poutcome_success,poutcome_unknown,age_group_31-40,age_group_41-50,age_group_51-60,age_group_60+,job_grouped_blue-collar,job_grouped_management,job_grouped_other,job_grouped_retired,job_grouped_self-employed,job_grouped_services,job_grouped_student,job_grouped_technician,job_grouped_unemployed,housing_yes,loan_yes
0,21,216,1,-1,0,0,5.379897,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,30,1599,2,-1,0,0,7.377759,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,61,967,1,-1,0,0,6.875232,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,30,3137,7,-1,0,0,8.051341,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4,35,3160,2,95,1,0,8.058644,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0


This step applies a conservative capping strategy to heavily skewed continuous features to prevent extreme outliers from distorting model training:

* **What it does:** Uses IQR-based winsorization (`fold=3`, applied to both tails) to cap extreme values in the `balance`, `campaign`, and `pdays` columns, storing the output in `x_train_capped`.
* **Why the method and columns match:** 
  * **Targeted Columns:** EDA flagged these three features as having severe right-skewness and massive ranges.
  * **Conservative Capping:** Utilizing `fold=3` instead of the standard `fold=1.5` ensures that only extreme, potentially malicious anomalies are clipped. This preserves genuine, high-value financial signals that a tree-based or linear model might still utilize.



### **1.4.4 Scaling**

In [ ]:
num_col = ["age", "balance", "campaign", "pdays", "balance_log"]

scaler = RobustScaler()

scaled_array = scaler.fit_transform(x_train_capped[num_col])

x_train_scaled = pd.DataFrame(scaled_array, columns=num_col, index=x_train.index)

display(x_train_scaled.head())

,age,balance,campaign,pdays,balance_log
0,-1.058824,-0.205488,-0.5,0.000000,-0.352178
1,-0.529412,0.677090,0.0,0.000000,0.421166
2,1.294118,0.273772,-0.5,0.000000,0.226645
3,-0.529412,1.658583,2.5,0.000000,0.681900
4,-0.235294,1.673261,0.0,2.666667,0.684727


This step standardizes the continuous numerical features to ensure they are on a comparable scale before being fed into the model:

* **What it does:** Uses `RobustScaler` to normalize five numeric variables: `age`, `balance`, `campaign`, `pdays`, and `balance_log`.
* **Why `RobustScaler` is chosen:**
  * **Outlier Resilience:** Unlike `StandardScaler` (mean/variance) or `MinMaxScaler` (minimum/maximum), `RobustScaler` relies on the **median** and **IQR**.
  * **Statistical Mitigation:** This prevents extreme anomalies from squeezing the majority of the data into a tiny range. It handles the heavy right-skew of the financial data through robust statistics rather than physical clipping.
* **Why Dummy Variables are Excluded:**
  * **Natural Scaling:** Binary flags (`0` or `1`) are already optimized for models. 
  * **Interpretability:** Scaling dummy variables would replace clear binary indicators with arbitrary decimal ranges without improving model performance, making final feature splits or weights much harder to analyze.


## **1.5 Feature Selection**

In [ ]:
x_train_final = x_train_encoded.copy()
x_train_final[num_col] = x_train_scaled[num_col]

print("Final Cleaned Training Data Shape:", x_train_final.shape)
display(x_train_final.head())


Final Cleaned Training Data Shape: (6244, 38)


,age,balance,campaign,pdays,pdays_contacted,balance_negative,balance_log,contact_telephone,contact_unknown,month_aug,month_dec,month_feb,month_jan,month_jul,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,poutcome_other,poutcome_success,poutcome_unknown,age_group_31-40,age_group_41-50,age_group_51-60,age_group_60+,job_grouped_blue-collar,job_grouped_management,job_grouped_other,job_grouped_retired,job_grouped_self-employed,job_grouped_services,job_grouped_student,job_grouped_technician,job_grouped_unemployed,housing_yes,loan_yes
0,-1.058824,-0.205488,-0.5,0.000000,0,0,-0.352178,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,-0.529412,0.677090,0.0,0.000000,0,0,0.421166,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,1.294118,0.273772,-0.5,0.000000,0,0,0.226645,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,-0.529412,1.658583,2.5,0.000000,0,0,0.681900,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4,-0.235294,1.673261,0.0,2.666667,1,0,0.684727,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0


This step merges the processed numeric and categorical features into a single, unified matrix optimized for model training:

* **What it does:** Directly overwrites the 5 continuous numeric columns within the one-hot encoded DataFrame (`x_train_encoded`) using their newly scaled values, yielding a final shape of **6,244 rows × 38 columns**.
* **Why this approach is used:** 
  * **In-Place Patching:** Modifying the existing encoded frame avoids the overhead of initializing a completely fresh DataFrame.
  * **Simplified Logic:** It bypasses redundant `pd.concat` or merging operations. The 33 binary dummy columns remain perfectly intact, while the numeric columns are cleanly updated with their scaled, outlier-resistant transformations.


In [ ]:
selector = SelectKBest(k=10)

x_train_selected_array = selector.fit_transform(x_train_final, y_train)
selected_columns = selector.get_feature_names_out()

# 5. Convert back to clean DataFrames
x_train_selected = pd.DataFrame(x_train_selected_array, columns=selected_columns, index=x_train_final.index)

print(f"Reduced features from {x_train_final.shape[1]} down to {x_train_selected.shape[1]}")
display(x_train_selected.head())

Reduced features from 38 down to 10


,pdays,pdays_contacted,balance_log,contact_unknown,month_may,month_oct,poutcome_success,poutcome_unknown,age_group_60+,housing_yes
0,0.000000,0.0,-0.352178,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,0.000000,0.0,0.421166,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,0.000000,0.0,0.226645,0.0,0.0,0.0,0.0,1.0,1.0,0.0
3,0.000000,0.0,0.681900,0.0,0.0,0.0,0.0,1.0,0.0,1.0
4,2.666667,1.0,0.684727,0.0,0.0,0.0,0.0,0.0,0.0,1.0


This step uses statistical ranking to identify the strongest individual predictors of the target variable and reduce feature dimensionality:

* **What it does:** Runs `SelectKBest` using the ANOVA F-test criterion to score all 38 features against `deposit` and filters the matrix down to the top 10 predictors.
* **The Selected Top 10 Features:** 
  * `pdays`, `pdays_contacted`, `balance_log`, `contact_unknown`, `month_may`, `month_oct`, `poutcome_success`, `poutcome_unknown`, `age_group_60+`, and `housing_yes`.
* **Why `k=10` is an Exploratory Step (Not a Constraint):**
  * **Statistical Limitation:** The ANOVA F-test scores features individually based on linear and marginal relationships, which means it can overlook columns that only become powerful when combined with other data.
  * **Grid Search Insights:** Downstream model tuning grids evaluated multiple feature boundaries (5, 10, 15, 20, and `'all'`) and found that keeping **`'all'`** features yielded the highest performance for tree ensembles like Gradient Boosting. 
  * **Purpose:** This cell should be treated as a useful diagnostic to see which variables carry the strongest standalone signal, rather than a definitive cutoff that restricts the training dataset.


In [ ]:
feature_scores = pd.DataFrame({
    'Feature': x_train_final.columns,
    'Score': selector.scores_
}).sort_values(by='Score', ascending=False)

display(feature_scores)

,Feature,Score
21,poutcome_success,564.537546
8,contact_unknown,465.127555
4,pdays_contacted,379.303056
22,poutcome_unknown,377.073171
36,housing_yes,274.287716
26,age_group_60+,183.006280
16,month_may,160.353153
3,pdays,158.354179
6,balance_log,141.677933
18,month_oct,129.589575


This step outputs the complete ANOVA F-test scores across all 38 features, providing a structured diagnostic ranking from highest to lowest predictive power:

* **What it does:** Displays a ranked list of univariate statistical scores for every feature against the `deposit` target variable.
* **Why this validates the EDA Pipeline:**
  * **Categorical Strengths:** The top ranking of `poutcome_success` (564.5) and `poutcome_unknown` (377.1) confirms the EDA finding where `poutcome` emerged as an exceptionally strong categorical predictor (Cramer's V ≈ 0.30). 
  * **Contact Insights:** `contact_unknown` (465.1) placing second echoes the EDA, which highlighted contact type as a vital predictive element.
  * **Feature Engineering Success:** The engineered feature `pdays_contacted` (379.3) ranks third overall. This proves that stripping the noise from raw `pdays` data into a simple binary indicator captures an intense, clean signal.
  * **Non-Linear Age Patterns:** `age_group_60+` (183.0) heavily outperforms other age groupings. This mathematically proves the EDA observation that conversion rates spike specifically for seniors rather than scaling linearly with age.
  * **Low-Signal Filtering:** The basement-level scores of features like `job_grouped_technician` (0.38) and `contact_telephone` (0.45) perfectly match the earlier Cramer's V ranking, which flagged the occupation categories as generally weak standalone predictors.
